# Notebook 5 — Comparações ML vs LLM

Protocolo Eduardo:
- Wilcoxon pareado seed-a-seed (bicaudal)
- Cohen's d (magnitude do efeito)
- Métricas: RMSE, R², MAE
- 20 seeds por modelo
- Targets: TOperacao, TAtracado

In [10]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from itertools import combinations
from scipy.stats import wilcoxon

warnings.filterwarnings('ignore')

METRICS_ML  = Path('metrics/ML_models')
METRICS_LLM = Path('metrics/LLM')

TARGET_COLS = ['TOperacao', 'TAtracado']
N_SEEDS     = 20

print('Config OK')

Config OK


## 1. Carregar resultados

In [11]:
# ── ML ────────────────────────────────────────────────────────────────────────
ml_files = {
    'xgboost':            METRICS_ML / 'results_xgboost.csv',
    'lightgbm':           METRICS_ML / 'results_lightgbm.csv',
    'random_forest':      METRICS_ML / 'results_random_forest.csv',
    'knn':                METRICS_ML / 'results_knn.csv',
    'linear_regression':  METRICS_ML / 'results_linear_regression.csv',
    'mlp':                METRICS_ML / 'results_mlp.csv',
    'svm':                METRICS_ML / 'results_svm.csv',
}

dfs_ml = []
for model, path in ml_files.items():
    if path.exists():
        df = pd.read_csv(path)
        df['group'] = 'ML'
        dfs_ml.append(df)
        print(f'  [OK] {model}: {len(df)} linhas')
    else:
        print(f'  [--] {model}: ainda não disponível')

# ── LLM ───────────────────────────────────────────────────────────────────────
llm_files = {
    'zero_shot': METRICS_LLM / 'results_zero_shot.csv',
    'few_shot':  METRICS_LLM / 'results_few_shot.csv',
}

dfs_llm = []
for strategy, path in llm_files.items():
    if path.exists():
        df = pd.read_csv(path)
        df['group'] = 'LLM'
        dfs_llm.append(df)
        print(f'  [OK] LLM {strategy}: {len(df)} linhas')
    else:
        print(f'  [--] LLM {strategy}: ainda não disponível')

df_ml  = pd.concat(dfs_ml,  ignore_index=True) if dfs_ml  else pd.DataFrame()
df_llm = pd.concat(dfs_llm, ignore_index=True) if dfs_llm else pd.DataFrame()
df_all = pd.concat([df_ml, df_llm], ignore_index=True)

print(f'\nTotal de linhas: {len(df_all)}')

  [OK] xgboost: 40 linhas
  [--] lightgbm: ainda não disponível
  [--] random_forest: ainda não disponível
  [OK] knn: 40 linhas
  [OK] linear_regression: 40 linhas
  [--] mlp: ainda não disponível
  [--] svm: ainda não disponível
  [OK] LLM zero_shot: 21 linhas
  [--] LLM few_shot: ainda não disponível

Total de linhas: 141


## 2. Tabela resumo — RMSE médio ± std por modelo e target

In [12]:
def summary_table(df, metric='rmse'):
    """RMSE mean ± std por modelo e target."""
    if df.empty:
        return pd.DataFrame()
    grp = df.groupby(['model', 'target'])[metric]
    mean = grp.mean().unstack('target')
    std  = grp.std().unstack('target')
    result = mean.copy()
    for col in mean.columns:
        result[col] = mean[col].map('{:.3f}'.format) + ' ± ' + std[col].map('{:.3f}'.format)
    return result

print('=== RMSE médio ± std (ML) ===')
display(summary_table(df_ml))

print('\n=== RMSE médio ± std (LLM) ===')
display(summary_table(df_llm))

=== RMSE médio ± std (ML) ===


target,TAtracado,TOperacao
model,,
knn,31.531 ± 0.000,17.073 ± 0.000
linear_regression,35.555 ± 0.372,17.786 ± 0.182
xgboost,20.304 ± 0.711,10.302 ± 0.419



=== RMSE médio ± std (LLM) ===


target,TAtracado,TOperacao
model,,
qwen3_32b,93.125 ± nan,54.317 ± 52.781


## 3. Wilcoxon pareado seed-a-seed + Cohen's d

Protocolo Eduardo: compara RMSE seed-a-seed entre pares de modelos.
Pareamento por índice de seed (seed 0 ML ↔ seed 0 LLM, etc.).

In [ ]:
def cohens_d(a, b):
    diff = np.array(a) - np.array(b)
    return float(diff.mean() / (diff.std(ddof=1) + 1e-12))


def get_metric_array(df, model, target, metric='rmse', strategy=None):
    """Retorna array de métrica ordenado por seed (20 valores)."""
    mask = (df['model'] == model) & (df['target'] == target)
    if strategy is not None:
        mask &= (df['strategy'] == strategy)
    sub = df[mask].sort_values('seed')
    return sub[metric].values


def wilcoxon_compare(df, model_a, model_b, target,
                     strategy_a=None, strategy_b=None, metric='rmse'):
    a = get_metric_array(df, model_a, target, metric, strategy_a)
    b = get_metric_array(df, model_b, target, metric, strategy_b)
    if len(a) < 2 or len(b) < 2 or len(a) != len(b):
        return None
    stat, p = wilcoxon(a, b)
    d = cohens_d(a, b)
    winner = model_a if a.mean() < b.mean() else model_b
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
    return {
        'metric': metric,
        'model_a': model_a, 'strategy_a': strategy_a or '-',
        'model_b': model_b, 'strategy_b': strategy_b or '-',
        'target': target,
        f'{metric}_a': round(a.mean(), 3), f'{metric}_b': round(b.mean(), 3),
        'p': round(p, 4), 'sig': sig,
        'd': round(d, 3), 'melhor': winner,
    }

print('Funções de teste OK')

### 3a. ML vs ML

In [ ]:
if not df_ml.empty:
    ml_models = df_ml['model'].unique().tolist()
    rows = []
    for metric in ['rmse', 'mae']:
        for ma, mb in combinations(ml_models, 2):
            for t in TARGET_COLS:
                r = wilcoxon_compare(df_ml, ma, mb, t, metric=metric)
                if r:
                    rows.append(r)
    df_wml = pd.DataFrame(rows)
    for metric in ['rmse', 'mae']:
        print(f'\n=== ML vs ML — {metric.upper()} ===')
        display(df_wml[df_wml['metric'] == metric].sort_values(['target', 'p']))
else:
    print('Sem dados ML ainda.')

### 3b. LLM vs LLM

In [ ]:
if not df_llm.empty:
    llm_combos = df_llm[['model', 'strategy']].drop_duplicates().values.tolist()
    rows = []
    for metric in ['rmse', 'mae']:
        for (ma, sa), (mb, sb) in combinations([tuple(x) for x in llm_combos], 2):
            for t in TARGET_COLS:
                r = wilcoxon_compare(df_llm, ma, mb, t, sa, sb, metric=metric)
                if r:
                    rows.append(r)
    if rows:
        df_wllm = pd.DataFrame(rows)
        for metric in ['rmse', 'mae']:
            print(f'\n=== LLM vs LLM — {metric.upper()} ===')
            display(df_wllm[df_wllm['metric'] == metric].sort_values(['target', 'p']))
    else:
        print('Poucos dados LLM ainda (precisa mais seeds/modelos).')
else:
    print('Sem dados LLM ainda.')

### 3c. ML vs LLM (melhor ML vs cada LLM)

In [7]:
if not df_ml.empty and not df_llm.empty:
    # Melhor ML por target = menor RMSE médio
    best_ml = (
        df_ml.groupby(['model', 'target'])['rmse']
        .mean()
        .reset_index()
        .sort_values('rmse')
        .groupby('target')
        .first()
        .reset_index()
    )
    print('Melhor ML por target:')
    display(best_ml)

    llm_combos = df_llm[['model', 'strategy']].drop_duplicates().values.tolist()
    rows = []
    for t in TARGET_COLS:
        best_model = best_ml[best_ml['target'] == t]['model'].values[0]
        for (lm, ls) in llm_combos:
            # Pareamento por índice (seed 0 ML ↔ seed 0 LLM)
            a = get_rmse_array(df_ml,  best_model, t)
            b = get_rmse_array(df_llm, lm, t, ls)
            min_len = min(len(a), len(b))
            if min_len < 2:
                continue
            a, b = a[:min_len], b[:min_len]
            stat, p = wilcoxon(a, b)
            d = cohens_d(a, b)
            sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
            rows.append({
                'target': t,
                'ml_model': best_model, 'rmse_ml': round(a.mean(), 3),
                'llm_model': lm, 'llm_strategy': ls, 'rmse_llm': round(b.mean(), 3),
                'p': round(p, 4), 'sig': sig, 'd': round(d, 3),
                'melhor': best_model if a.mean() < b.mean() else f'{lm}/{ls}',
            })

    if rows:
        df_wcomp = pd.DataFrame(rows)
        display(df_wcomp.sort_values(['target', 'p']))
    else:
        print('Dados insuficientes para comparação ML vs LLM.')
else:
    print('Aguardando dados ML e LLM.')

Melhor ML por target:


,target,model,rmse
0,TAtracado,xgboost,20.303561
1,TOperacao,xgboost,10.301820


,target,ml_model,rmse_ml,llm_model,llm_strategy,rmse_llm,p,sig,d,melhor
0,TOperacao,xgboost,10.312,qwen3_32b,zero_shot,58.581,0.002,**,-0.77,xgboost


## 4. Ranking geral por target

In [ ]:
if not df_all.empty:
    df_all['strategy'] = df_all.get('strategy', pd.Series('', index=df_all.index)).fillna('-')

    ranking = (
        df_all.groupby(['model', 'strategy', 'group', 'target'])[['rmse', 'r2', 'mae']]
        .agg(['mean', 'std'])
        .round(3)
        .reset_index()
    )
    ranking.columns = [
        'model', 'strategy', 'group', 'target',
        'rmse_mean', 'rmse_std', 'r2_mean', 'r2_std', 'mae_mean', 'mae_std'
    ]

    for t in TARGET_COLS:
        sub = ranking[ranking['target'] == t].sort_values('rmse_mean').reset_index(drop=True)
        print(f'\n=== Ranking — {t} ===')
        display(sub[['model', 'strategy', 'group', 'rmse_mean', 'rmse_std', 'r2_mean', 'mae_mean']])
else:
    print('Sem dados ainda.')

## 5. Tabela final para o paper

In [9]:
if not df_all.empty:
    def make_label(row):
        s = row.get('strategy', None)
        return row['model'] if pd.isna(s) or s == '-' else f"{row['model']} ({s})"

    df_all['label'] = df_all.apply(make_label, axis=1)

    for t in TARGET_COLS:
        sub = df_all[df_all['target'] == t]
        tbl = (
            sub.groupby('label')[['rmse', 'r2', 'mae']]
            .agg(['mean', 'std'])
            .round(3)
        )
        tbl.columns = ['RMSE_mean', 'RMSE_std', 'R2_mean', 'R2_std', 'MAE_mean', 'MAE_std']
        tbl['RMSE'] = tbl['RMSE_mean'].map('{:.3f}'.format) + ' ± ' + tbl['RMSE_std'].map('{:.3f}'.format)
        tbl['R²']   = tbl['R2_mean'].map('{:.4f}'.format)  + ' ± ' + tbl['R2_std'].map('{:.4f}'.format)
        tbl['MAE']  = tbl['MAE_mean'].map('{:.3f}'.format) + ' ± ' + tbl['MAE_std'].map('{:.3f}'.format)
        tbl = tbl[['RMSE', 'R²', 'MAE']].sort_values('RMSE')
        print(f'\n=== {t} ===')
        display(tbl)
else:
    print('Sem dados ainda.')


=== TOperacao ===


,RMSE,R²,MAE
label,,,
xgboost,10.302 ± 0.419,0.8940 ± 0.0090,3.444 ± 0.110
knn,17.073 ± 0.000,0.7080 ± 0.0000,8.195 ± 0.000
linear_regression,17.786 ± 0.182,0.6830 ± 0.0060,8.493 ± 0.107
qwen3_32b (zero_shot),58.581 ± 62.626,-5.9720 ± 14.9490,14.346 ± 5.011



=== TAtracado ===


,RMSE,R²,MAE
label,,,
xgboost,20.304 ± 0.711,0.7150 ± 0.0200,3.865 ± 0.120
knn,31.531 ± 0.000,0.3120 ± 0.0000,11.756 ± 0.000
linear_regression,35.555 ± 0.372,0.1260 ± 0.0180,10.590 ± 0.117
